In [ ]:
import json
import pandas as pd
from pathlib import Path
DATA_PATH = DATA_PATH = Path(r"C:\Users\Eric Arnold\Documents\TCGA_data")

In [13]:
with open(DATA_PATH / "metadata.repository.2026-06-04.json") as f:
    metadata = json.load(f)

slides = []
for entry in metadata:
    for entity in entry["associated_entities"]:
        slides.append({
            "patient_name": entity["entity_submitter_id"],
            "sample_id": entity["entity_submitter_id"][:15],
            "entity_id": entity["entity_id"],
            "file_id": entry["file_id"],
            "file_name": entry["file_name"]
        })

slides_df = pd.DataFrame(slides)

clinical = pd.read_csv(DATA_PATH / "TCGA.BRCA.sampleMap_BRCA_clinicalMatrix", sep="\t")
pam50 = clinical[["sampleID", "PAM50Call_RNAseq"]].dropna()
pam50 = pam50[pam50["PAM50Call_RNAseq"] != "Normal"]

result = slides_df.merge(pam50, left_on="sample_id", right_on="sampleID", how="inner")
result = result[["patient_name", "entity_id", "file_id", "PAM50Call_RNAseq"]].rename(columns={"PAM50Call_RNAseq": "subtype"})

print(result)
print(result["subtype"].value_counts())

                patient_name                             entity_id  \
0    TCGA-E2-A14P-01Z-00-DX1  6f9f59be-f550-4f53-8d7f-9f96fe1db152   
1    TCGA-A7-A0CD-01Z-00-DX1  2c72ef33-b4d7-406b-9b5a-8f1cf8cd1225   
2    TCGA-A8-A09K-01Z-00-DX1  cb67baac-9b6c-4ef9-868d-25d34536877a   
3    TCGA-C8-A1HI-01Z-00-DX1  547b78b6-0973-48b8-82cf-1998f42fe005   
4    TCGA-AQ-A0Y5-01Z-00-DX1  ca1dd290-465b-4747-9760-e8bf1df8295f   
..                       ...                                   ...   
836  TCGA-BH-A1F2-01Z-00-DX1  a474a1b7-82f5-46f5-9db1-f4b93c9d47be   
837  TCGA-A7-A13G-01Z-00-DX1  7605cd2c-6de4-4956-b2f0-dd59ea834685   
838  TCGA-GM-A2D9-01Z-00-DX1  b45102e1-01f3-49cd-a0a8-acd284c18605   
839  TCGA-D8-A1XS-01Z-00-DX1  9e7525e8-cb1c-4ec6-847d-ef22a6d57a67   
840  TCGA-D8-A1XS-01Z-00-DX2  2231fda6-e015-4873-8f58-4553908fb09e   

                                  file_id subtype  
0    4730b23e-aea1-49a2-ba63-2231fd88b592    Her2  
1    554855d7-4e21-406b-8f9f-458b1e7c89c9    LumA  
2  

In [ ]:
# Proportional sampling of 200 slides
total = 200
class_counts = result["subtype"].value_counts()
class_proportions = (class_counts / class_counts.sum() * total).round().astype(int)
diff = total - class_proportions.sum()
class_proportions.iloc[0] += diff

sampled = pd.concat([
    group.sample(n=class_proportions[name], random_state=42)
    for name, group in result.groupby("subtype")
]).reset_index(drop=True)

print(f"Total sampled: {len(sampled)}")
print(sampled["subtype"].value_counts())

full_manifest = pd.read_csv(DATA_PATH / "gdc_manifest.tsv.txt", sep="\t")
manifest_200 = full_manifest[full_manifest["id"].isin(sampled["file_id"])]
print(f"Matched in manifest: {len(manifest_200)}")
manifest_200.to_csv(DATA_PATH / "manifest_200.tsv", sep="\t", index=False)

Total sampled: 200
subtype
LumA     101
LumB      51
Basal     33
Her2      15
Name: count, dtype: int64
Matched in manifest: 200
